# 04. 장르별 성과 비교 — 다중 장르 중복 집계 방식

**분석 목적:** 한 게임이 가진 모든 장르에 중복으로 집계하여 장르별 긍정률·리뷰 수·가격을 비교한다.

**사용 데이터:** `data/preprocessed/steam_indie_games.csv` (9,692개, 리뷰 10개 이상, 2023~2025년, EA·F2P 제외)

**분석 흐름:**
1. 데이터 로드 및 장르 explode
2. 장르별 게임 수 분포
3. 장르별 리뷰 수 분포 (시장 규모)
4. 장르별 긍정률 분포 (유저 만족도)
5. 장르별 가격 분포
6. 긍정률 × 리뷰 수 포지셔닝 산점도
7. 종합 요약 및 인사이트

---

## 분석 방법론 및 한계

### 다중 장르 집계 방식을 선택한 이유

실제 Steam 인디게임의 대부분은 복합 장르를 가진다. 게임 하나에 대표 장르를 하나만 부여하면 나머지 장르 속성이 무시되고, 어떤 기준으로 대표 장르를 정하든 자의적인 판단이 개입된다. 따라서 게임이 가진 모든 장르에 중복 집계하는 방식을 선택했다.

이 방식은 **"해당 장르 속성을 가진 게임들의 성과"** 를 분석하는 질문에 적합하다. 인디 개발사가 "어떤 장르를 선택할까"를 고민할 때 실제로 참고하는 맥락과 일치한다.

### 방법론의 한계

| 한계 | 설명 |
|------|------|
| 중복 집계 | 동일 게임이 여러 장르에 포함되어 장르 간 독립성이 없다 |
| 표본 과장 | 장르별 n수가 실제 고유 게임 수보다 많아 통계적 유의성이 과장될 수 있다 |
| 조합 효과 미반영 | "Action+RPG 조합"의 성과와 "Action 단독"의 성과를 구분하지 않는다 |

따라서 본 분석의 결과는 **장르별 절대적 성과 비교가 아닌, 해당 장르 속성을 포함한 게임군의 경향성**으로 해석해야 한다.

## 0. 라이브러리 로드

In [28]:
import ast
import warnings

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')

TARGET_GENRES = ['Action', 'Adventure', 'Casual', 'RPG', 'Simulation', 'Strategy', 'Sports', 'Racing']

PALETTE = [
    '#4C72B0', '#DD8452', '#55A868', '#C44E52',
    '#8172B2', '#937860', '#DA8BC3', '#8C8C8C'
]
COLOR_MAP = {g: PALETTE[i] for i, g in enumerate(TARGET_GENRES)}

## 1. 데이터 로드 및 장르 explode

In [29]:
games = pd.read_csv('../../data/preprocessed/steam_indie_games.csv')
print(f'전체 게임: {len(games):,}개')
games.head(3)

전체 게임: 9,692개


,appid,positive,negative,price,genres,total_reviews,name,developers,release_date,short_description,...,categories,windows,mac,linux,recommendations_total,achievements_total,owners_lower,owners_higher,tags,updated_at
0,226620,1912,364,14.99,"['Adventure', 'Casual', 'Indie', 'RPG', 'Strat...",2276,Desktop Dungeons,QCF Design,2023-04-18,Each step into the unknown heals you and revea...,...,"Single-player, Steam Achievements, Steam Tradi...",True,True,True,1129.0,35.0,200000,500000,"{""2D"": 36, ""RPG"": 103, ""Dwarf"": 18, ""Casual"": ...",2026-04-27 20:02:19.802355
1,230210,303,45,24.99,"['Adventure', 'Indie']",348,ASYLUM,Senscape,2025-03-13,An epic supernatural horror adventure and the ...,...,"Single-player, Steam Achievements, Full contro...",True,True,False,392.0,19.0,0,20000,"{""Dark"": 40, ""Gore"": 52, ""Indie"": 76, ""Gothic""...",2026-04-27 21:13:17.714243
2,251570,327889,42157,44.99,"['Action', 'Adventure', 'Indie', 'RPG', 'Simul...",370046,7 Days to Die,The Fun Pimps,2024-07-25,7 Days to Die is an open-world game that is a ...,...,"Single-player, Multi-player, PvP, Online PvP, ...",True,True,True,271581.0,43.0,10000000,20000000,"{""FPS"": 3827, ""Voxel"": 4264, ""Action"": 3694, ""...",2026-04-27 19:17:36.995662


In [30]:
games['genres_list'] = games['genres'].apply(lambda g: ast.literal_eval(g) if pd.notna(g) else [])
games['positive_rate'] = games['positive'] / games['total_reviews'] * 100
games['genres_filtered'] = games['genres_list'].apply(
    lambda gl: [g for g in gl if g in TARGET_GENRES]
)

games_with_genre = games[games['genres_filtered'].map(len) > 0].copy()
df_multi = games_with_genre.explode('genres_filtered').rename(columns={'genres_filtered': 'genre'})
df_multi_paid = df_multi[(df_multi['price'] > 0) & (df_multi['price'] <= 60)].copy()

print(f'원본 게임 수    : {len(games_with_genre):,}개')
print(f'explode 후 행 수: {len(df_multi):,}행 (중복 포함)')
print()
print('장르별 집계 게임 수:')
print(df_multi['genre'].value_counts().to_string())

원본 게임 수    : 9,314개
explode 후 행 수: 20,355행 (중복 포함)

장르별 집계 게임 수:
genre
Adventure     4807
Casual        4111
Action        4093
Simulation    2503
RPG           2194
Strategy      2005
Sports         341
Racing         301


**다중 장르 집계 방식:** 한 게임이 복수 장르를 가지면 각 장르 집합에 모두 포함된다. 전체 행 수가 원본 게임 수보다 많으며 장르별 게임 수의 합이 전체 게임 수를 초과한다. 동일 게임이 중복 집계된다는 점을 해석 시 감안해야 한다.

## 2. 장르별 게임 수

In [31]:
genre_counts = df_multi['genre'].value_counts().reset_index()
genre_counts.columns = ['genre', 'count']

fig = px.bar(
    genre_counts,
    x='genre', y='count',
    color='genre',
    color_discrete_map=COLOR_MAP,
    text='count',
    title='장르별 게임 수 (다중 장르 중복 집계)',
    labels={'genre': '장르', 'count': '게임 수 (중복 포함)'},
)
fig.update_traces(texttemplate='%{text:,}', textposition='outside')
fig.update_layout(showlegend=False, xaxis_categoryorder='total descending')
fig.show()

**해석:** 다중 장르 방식에서 Adventure·Action·Casual이 가장 많다. RPG·Simulation·Strategy는 단독보다 Action·Adventure와 함께 조합되는 경우가 많아 이 방식에서 상당한 수를 차지한다.

## 3. 장르별 리뷰 수 분포 (시장 규모)

In [32]:
genre_order_reviews = (
    df_multi.groupby('genre')['total_reviews']
    .median()
    .sort_values(ascending=False)
    .index.tolist()
)

fig = px.box(
    df_multi,
    x='genre', y='total_reviews',
    category_orders={'genre': genre_order_reviews},
    color='genre',
    color_discrete_map=COLOR_MAP,
    points=False,
    title='장르별 총 리뷰 수 분포 (다중 장르 방식)',
    labels={'genre': '장르', 'total_reviews': '총 리뷰 수 (log scale)'},
)
fig.update_layout(showlegend=False, yaxis_type='log')
fig.show()

**해석:** RPG·Simulation·Strategy는 게임 수가 상대적으로 적음에도 게임당 리뷰 수 중앙값이 높아 공급 대비 유저 수요가 큰 장르다. 반면 Casual은 출시 게임 수 1위임에도 리뷰 수 중앙값이 낮아 공급 과잉 구조가 두드러진다.

In [33]:
df_multi.groupby('genre')['total_reviews'].agg(['count', 'median', 'mean', 'std']).round(1).sort_values('median', ascending=False)

,count,median,mean,std
genre,,,,
RPG,2194,70.5,1303.5,10767.0
Simulation,2503,57.0,1338.8,12183.4
Strategy,2005,52.0,1098.4,10849.9
Adventure,4807,44.0,990.7,9846.8
Action,4093,38.0,1092.3,10285.8
Sports,341,38.0,265.5,1644.0
Casual,4111,35.0,488.3,3933.5
Racing,301,29.0,665.2,5604.0


**해석:** 리뷰 수 중앙값이 높은 장르는 해당 속성을 가진 게임들이 전반적으로 더 많은 유저 참여를 이끌어낸다는 의미다. RPG·Simulation·Strategy가 중앙값 기준 상위권인데, 이 장르들은 게임당 플레이타임이 길고 몰입도 높은 유저층을 보유하는 경향이 있다.

## 4. 장르별 긍정률 분포 (유저 만족도)

In [34]:
genre_order_pos = (
    df_multi.groupby('genre')['positive_rate']
    .median()
    .sort_values(ascending=False)
    .index.tolist()
)

fig = px.box(
    df_multi,
    x='genre', y='positive_rate',
    category_orders={'genre': genre_order_pos},
    color='genre',
    color_discrete_map=COLOR_MAP,
    points=False,
    title='장르별 긍정률 분포 (다중 장르 방식)',
    labels={'genre': '장르', 'positive_rate': '긍정률 (%)'},
)
fig.add_hline(y=80, line_dash='dash', line_color='red',
              annotation_text='80% (Very Positive)', annotation_position='top right')
fig.update_layout(showlegend=False, yaxis_range=[0, 110])
fig.show()

**해석:** 전 장르 긍정률 중앙값이 80% 이상으로 높고, 장르 간 절대 격차가 크지 않다. 장르 선택이 초기 유저 만족도를 결정짓는 주요 요인이 아님을 시사하며, 시장의 병목은 품질보다 발견(노출) 측면에 있다.

In [35]:
df_multi.groupby('genre')['positive_rate'].agg(['count', 'median', 'mean', 'std']).round(2).sort_values('median', ascending=False)

,count,median,mean,std
genre,,,,
Casual,4111,89.71,85.06,15.22
Action,4093,87.50,83.39,15.87
Adventure,4807,87.43,83.39,15.30
Racing,301,87.07,82.36,16.41
Strategy,2005,86.26,82.93,14.77
RPG,2194,85.71,82.23,15.11
Sports,341,85.28,82.52,15.28
Simulation,2503,83.46,79.46,16.81


**해석:** 모든 장르의 긍정률 중앙값이 80% 이상으로 Steam 'Very Positive' 수준을 유지하고 있다. Casual이 가장 높고 Simulation이 가장 낮다. Simulation의 std가 16.81로 가장 커서 품질 편차가 크다는 점도 주목할 만하다.

## 5. 장르별 가격 분포

In [36]:
genre_order_price = (
    df_multi_paid.groupby('genre')['price']
    .median()
    .sort_values(ascending=False)
    .index.tolist()
)

fig = px.box(
    df_multi_paid,
    x='genre', y='price',
    category_orders={'genre': genre_order_price},
    color='genre',
    color_discrete_map=COLOR_MAP,
    points=False,
    title='장르별 출시 가격 분포 (유료 게임, 다중 장르 방식)',
    labels={'genre': '장르', 'price': '가격 (USD)'},
)
fig.update_layout(showlegend=False)
fig.update_yaxes(tickprefix='$')
fig.show()

**해석:** 유료 게임 기준 장르별 가격 분포를 확인한다. RPG·Strategy는 타 장르 대비 중앙값 가격이 높은 편이며, Casual·Action은 저가 구간에 집중된다. 전반적으로 인디게임 시장은 $10 이하 저가 구간에 게임이 집중되는 구조다.

In [37]:
df_multi_paid.groupby('genre')['price'].agg(['count', 'median', 'mean', 'std']).round(2).sort_values('median', ascending=False)

,count,median,mean,std
genre,,,,
RPG,2140,8.99,10.33,7.61
Strategy,1954,7.99,10.01,7.52
Action,3999,6.99,8.96,7.20
Adventure,4699,6.99,9.14,7.33
Simulation,2426,6.99,9.21,7.67
Sports,324,6.99,9.81,8.98
Casual,4006,4.99,7.32,6.28
Racing,291,4.99,8.13,6.99


In [38]:
# 가격대 구간 정의
bins   = [0, 5, 10, 20, 30, 60]
labels = ['~$5', '$5~10', '$10~20', '$20~30', '$30~60']

df_multi_paid['price_range'] = pd.cut(df_multi_paid['price'], bins=bins, labels=labels, right=True)

price_range_stats = (
    df_multi_paid.groupby('price_range', observed=True)
    .agg(
        게임수=('appid', 'nunique'),
        긍정률_중앙값=('positive_rate', 'median'),
        긍정률_평균=('positive_rate', 'mean'),
        리뷰수_중앙값=('total_reviews', 'median'),
    )
    .round(2)
    .reset_index()
)

display(price_range_stats)

,price_range,게임수,긍정률_중앙값,긍정률_평균,리뷰수_중앙값
0,~$5,3975,88.24,83.33,27.0
1,$5~10,2582,87.93,83.82,44.0
2,$10~20,2133,86.46,83.20,122.0
3,$20~30,350,83.33,81.05,471.0
4,$30~60,72,82.56,81.05,583.5


In [39]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('가격대별 게임 수', '가격대별 긍정률 중앙값'),
)

fig.add_trace(go.Bar(
    x=price_range_stats['price_range'],
    y=price_range_stats['게임수'],
    text=price_range_stats['게임수'],
    textposition='outside',
    marker_color='#4C72B0',
    showlegend=False,
), row=1, col=1)

fig.add_trace(go.Bar(
    x=price_range_stats['price_range'],
    y=price_range_stats['긍정률_중앙값'],
    text=price_range_stats['긍정률_중앙값'].apply(lambda x: f'{x:.1f}%'),
    textposition='outside',
    marker_color='#55A868',
    showlegend=False,
), row=1, col=2)

fig.add_hline(y=80, line_dash='dash', line_color='red', opacity=0.6,
              annotation_text='80%', annotation_position='top right', row=1, col=2)

fig.update_layout(title='가격대별 게임 수 및 긍정률', height=420)
fig.update_yaxes(title_text='게임 수', row=1, col=1)
fig.update_yaxes(title_text='긍정률 중앙값 (%)', range=[0, 105], row=1, col=2)
fig.update_xaxes(title_text='가격대')
fig.show()

**해석:** RPG·Strategy가 가격 중앙값이 높고 Casual·Racing이 낮다. 신규 인디 개발사라면 해당 장르의 중앙값 ±20% 범위를 출시 가격 기준점으로 삼는 것을 권장한다. 시장 형성 가격대를 크게 벗어나면 구매 전환에 불리하게 작용할 수 있다.

## 6. 긍정률 × 리뷰 수 포지셔닝 산점도

In [40]:
# 장르별 중앙값 계산
medians = df_multi.groupby('genre').agg(
    리뷰수_중앙값=('total_reviews', 'median'),
    긍정률_중앙값=('positive_rate', 'median'),
    게임수=('appid', 'count'),
).reset_index()

fig = go.Figure()

# 개별 게임 산점도
for genre in TARGET_GENRES:
    subset = df_multi[df_multi['genre'] == genre]
    if len(subset) == 0:
        continue
    fig.add_trace(go.Scatter(
        x=subset['total_reviews'],
        y=subset['positive_rate'],
        mode='markers',
        name=genre,
        marker=dict(color=COLOR_MAP[genre], size=5, opacity=0.35),
        legendgroup=genre,
        showlegend=True,
        hovertemplate='%{customdata}<br>리뷰: %{x:,}<br>긍정률: %{y:.1f}%<extra>' + genre + '</extra>',
        customdata=subset['name'],
    ))

# 장르별 중앙값 마커
for _, row in medians.iterrows():
    genre = row['genre']
    fig.add_trace(go.Scatter(
        x=[row['리뷰수_중앙값']],
        y=[row['긍정률_중앙값']],
        mode='markers+text',
        marker=dict(color=COLOR_MAP.get(genre, '#8C8C8C'), size=14, symbol='diamond',
                    line=dict(color='black', width=1)),
        text=[genre],
        textposition='top right',
        textfont=dict(size=11, color=COLOR_MAP.get(genre, '#8C8C8C')),
        legendgroup=genre,
        showlegend=False,
        hovertemplate=f'{genre}<br>중앙값 리뷰: {int(row["리뷰수_중앙값"]):,}<br>중앙값 긍정률: {row["긍정률_중앙값"]:.1f}%<extra>중앙값</extra>',
    ))

# 80% 기준선
fig.add_hline(y=80, line_dash='dash', line_color='gray', opacity=0.6,
              annotation_text='80% 기준선', annotation_position='top right')

fig.update_layout(
    title='장르별 포지셔닝: 리뷰 수(규모) × 긍정률(만족도)<br><sub>다이아몬드 = 장르 중앙값 / 다중 장르 중복 집계</sub>',
    xaxis=dict(title='총 리뷰 수 (log scale)', type='log'),
    yaxis=dict(title='긍정률 (%)', range=[20, 108]),
    legend=dict(title='장르'),
    height=600,
)
fig.show()

**해석:**
- **우상단 (리뷰 많음 + 긍정률 높음):** 시장이 크고 만족도도 높은 장르 — 경쟁 치열, 성공 시 파급력 큼
- **좌상단 (리뷰 적음 + 긍정률 높음):** 틈새 시장, 핵심 팬층 만족 — 마케팅 강화 시 평판 확보 가능
- **우하단 (리뷰 많음 + 긍정률 낮음):** 기대치 대비 실망이 큰 장르 — 품질 차별화 필수
- **좌하단 (리뷰 적음 + 긍정률 낮음):** 시장·만족도 모두 약한 장르 — 신중한 접근 필요

## 7. 종합 요약 테이블 및 인사이트

In [41]:
summary = df_multi.groupby('genre').agg(
    게임수=('appid', 'count'),
    리뷰수_중앙값=('total_reviews', 'median'),
    긍정률_중앙값=('positive_rate', 'median'),
    긍정률_std=('positive_rate', 'std'),
).round(2)

price_med = df_multi_paid.groupby('genre')['price'].median().rename('가격_중앙값_USD')
summary = summary.join(price_med)
summary['리뷰수_중앙값'] = summary['리뷰수_중앙값'].astype(int)
summary = summary.sort_values('긍정률_중앙값', ascending=False)

display(summary)

,게임수,리뷰수_중앙값,긍정률_중앙값,긍정률_std,가격_중앙값_USD
genre,,,,,
Casual,4111,35,89.71,15.22,4.99
Action,4093,38,87.50,15.87,6.99
Adventure,4807,44,87.43,15.30,6.99
Racing,301,29,87.07,16.41,4.99
Strategy,2005,52,86.26,14.77,7.99
RPG,2194,70,85.71,15.11,8.99
Sports,341,38,85.28,15.28,6.99
Simulation,2503,57,83.46,16.81,6.99


### 인디 개발사를 위한 장르 선택 가이드

| 상황 | 추천 전략 |
|------|----------|
| 초기작, 리소스 제한 | 긍정률 중앙값 높고 가격 중앙값 낮은 장르 — 진입 장벽 낮음 |
| 시장 규모 우선 | 리뷰 수 중앙값 높은 장르 — 단, 품질 차별화 필수 |
| 고평가 목표 | 긍정률 중앙값 85% 이상 장르 + 상위 게임 벤치마킹 |
| 틈새 공략 | 리뷰 수 적지만 긍정률 높은 장르 — 커뮤니티 마케팅 병행 |